In [1]:
import BioSimSpace as BSS

import pandas as pd

from pathlib import Path

import re

# on this occasion we will not be using a node, instead we will use a function
from nodes.fep_prep import prepare_fep_free, prepare_fep_bound
from dask.distributed import Client, LocalCluster, wait, as_completed

INFO:rdkit:Enabling RDKit 2024.03.5 jupyter extensions
INFO:numexpr.utils:Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [2]:
def get_ligand_names(filename):
    p = Path(filename).stem
    p = p.split("_")[0]
    return p

In [3]:
# we will grab the engine and runtime from our protocol file
config = {}
# read config
with open("output_setup/protocol.dat","r") as file:
    for line in file:
        key, value = line.split("=")
        config[str(key).strip()] = str(value).strip().replace('*','')

In [4]:
runtime = BSS.Types.Time(config["sampling"]).nanoseconds()

In [5]:
# read the network file to find our perturbations
df_network = pd.read_csv("output_setup/network.dat",sep='\s+', names=["lig1","lig2","num_windows","lambda_vals","engine"])

In [6]:
free_folder = Path("equilibrated_free_systems")
free_files = list(free_folder.glob("*.prm7")) + list(free_folder.glob("*.rst7"))

bound_folder = Path("equilibrated_bound_systems")
bound_files = list(bound_folder.glob("*.prm7")) + list(bound_folder.glob("*.rst7"))

In [7]:
# files for free legs
ligand_names_locations_free = {}
for files in free_files:
    name = get_ligand_names(files)
    if name in ligand_names_locations_free.keys():
        ligand_names_locations_free[name].append(str(files.absolute()))
    else:
        ligand_names_locations_free[name] = [str(files.absolute())]

# files for bound legs
ligand_names_locations_bound = {}
for files in bound_files:
    name = get_ligand_names(files)
    if name in ligand_names_locations_bound.keys():
        ligand_names_locations_bound[name].append(str(files.absolute()))
    else:
        ligand_names_locations_bound[name] = [str(files.absolute())]

In [8]:
inputs_for_free_runs = []
# now we loop over the given perturbations and create inputs for our prep function
for row in df_network.iterrows():
    row = row[1]
    lig1 = row["lig1"]
    lig2 = row["lig2"]
    if (lig1 not in ligand_names_locations_free.keys()) or (lig2 not in ligand_names_locations_free.keys()):
        print(f"ligands not found in equilibrated free systems ({lig1} or {lig2})")
        continue
    lambds = [float(i) for i in row["lambda_vals"].strip().split(",")]
    #engine= config["engine"]
    #settings for testing
    engine = "SOMD"
    runtime = "1ps"
    inp = {
           "ligand1_name":lig1,
           "ligand2_name":lig2,
           "ligand1_files":ligand_names_locations_free[lig1], 
           "ligand2_files":ligand_names_locations_free[lig2],
           "output_location":Path(f"production/{engine}/").absolute(),
           "num_lambda":len(lambds),
           "lambda_values":lambds,
           "md_engine":engine,
           "runtime":runtime,
          }
    inputs_for_free_runs.append(inp)

ligands not found in equilibrated free systems (ejm42 or ejm49)
ligands not found in equilibrated free systems (ejm49 or ejm42)
ligands not found in equilibrated free systems (ejm48 or ejm49)
ligands not found in equilibrated free systems (ejm49 or ejm48)


In [9]:
inputs_for_bound_runs = []
# now we loop over the given perturbations and create inputs for our prep function
for row in df_network.iterrows():
    row = row[1]
    lig1 = row["lig1"]
    lig2 = row["lig2"]
    if (lig1 not in ligand_names_locations_bound.keys()) or (lig2 not in ligand_names_locations_bound.keys()):
        print(f"ligands not found in equilibrated bound systems ({lig1} or {lig2})")
        continue
    lambds = [float(i) for i in row["lambda_vals"].strip().split(",")]
    #engine= config["engine"]
    #settings for testing
    engine = "SOMD"
    runtime = "1ps"
    inp = {
           "ligand1_name":lig1,
           "ligand2_name":lig2,
           "ligand1_files":ligand_names_locations_bound[lig1], 
           "ligand2_files":ligand_names_locations_bound[lig2],
           "output_location":Path(f"production/{engine}/").absolute(),
           "num_lambda":len(lambds),
           "lambda_values":lambds,
           "md_engine":engine,
           "runtime":runtime,
          }
    inputs_for_bound_runs.append(inp)

ligands not found in equilibrated bound systems (ejm42 or ejm49)
ligands not found in equilibrated bound systems (ejm49 or ejm42)
ligands not found in equilibrated bound systems (ejm48 or ejm49)
ligands not found in equilibrated bound systems (ejm49 or ejm48)


In [10]:
# Now we can create a localcluster to setup our simulation network.
# If using `setup_only=False` it is strongly recommended to use HPC resources as running the full network will be very computationally intensive
cluster = LocalCluster(
    n_workers=4,
    memory_limit="5000MB",
    threads_per_worker = 5,
    # We will use CPU_task to ensure that only a single parameterisation/solvation is performed per worker
    resources={"CPU_task": 1},
)

INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:33445
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43251'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43929'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:42577'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:35451'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:34581 name: 2
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:34581
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:51898
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:35737 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:35737
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:51

In [11]:
client = Client(cluster)
#node_plugin = SetNodeDir(node_path)
#client.register_plugin(node_plugin, name="node-setup")
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 20,Total memory: 18.63 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33445,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 20
Started: Just now,Total memory: 18.63 GiB
Comm: tcp://127.0.0.1:42309,Total threads: 5
Dashboard: http://127.0.0.1:36465/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:43251,


In [12]:
futures = [client.submit(prepare_fep_free, **inp, resources={"CPU_task":1}) for inp in inputs_for_free_runs]

In [13]:
for f in as_completed(futures):
    _ = f.result()
    f.release()
    client.cancel(f)

In [14]:
client.restart()

In [16]:
futures = [client.submit(prepare_fep_bound, **inp, resources={"CPU_task":1}) for inp in inputs_for_bound_runs]

2025-04-01 14:35:18,323 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('prepare_fep_bound-98d9d801bf22b8d5d9df168210aa4f02')" coro=<Worker.execute() done, defined at /home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-04-01 14:35:18,793 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('prepare_fep_bound-1f49a98715b101af505964ee371b176b')" coro=<Worker.execute() done, defined at /home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 futures = [client.submit(prepare_fep_bound, **inp, resources={"CPU_task":1}) for inp in      │
│ ❱ 2 _ = wait(futures)                                                                            │
│   3                                                                                              │
│                                                                                                  │
│ /home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/client.py:5714 in    │
│ wait                                                                                             │
│                                                                                                  │
│   5711 │   if timeout is not None and isinstance(timeout, (Number, str)):                        │
│   5712 │   │   timeout = parse_timedelta(timeout, default="s")                                   │
│   5713 │   client = default_client()                                                             │
│ ❱ 5714 │   result = client.sync(_wait, fs, timeout=timeout, return_when=return_when)             │
│   5715 │   return result                                                                         │
│   5716                                                                                           │
│   5717                                                                                           │
│                                                                                                  │
│ /home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/utils.py:363 in sync │
│                                                                                                  │
│    360 │   │   │   │   future = wait_for(future, callback_timeout)                               │
│    361 │   │   │   return future                                                                 │
│    362 │   │   else:                                                                             │
│ ❱  363 │   │   │   return sync(                                                                  │
│    364 │   │   │   │   self.loop, func, *args, callback_timeout=callback_timeout, **kwargs       │
│    365 │   │   │   )                                                                             │
│    366                                                                                           │
│                                                                                                  │
│ /home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/utils.py:436 in sync │
│                                                                                                  │
│    433 │   │   │   raise TimeoutError(f"timed out after {timeout} s.")                           │
│    434 │   else:                                                                                 │
│    435 │   │   while not e.is_set():                                                             │
│ ❱  436 │   │   │   wait(10)                                                                      │
│    437 │                                                                                         │
│    438 │   if error is not None:                                                                 │
│    439 │   │   raise error                                                                       │
│                                                                                                  │
│ /home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/utils.py:425 in wait │
│                                                            

2025-04-01 14:35:19,434 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('prepare_fep_bound-e9a0ddcebce007927c3a8eaa0c62e88a')" coro=<Worker.execute() done, defined at /home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-04-01 14:35:19,759 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('prepare_fep_bound-01fec45483303b36a374a526777a2891')" coro=<Worker.execute() done, defined at /home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-04-01 14:35:20,490 - distributed.worker - ERROR - 
Traceback (most recent call last):
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/compatibility.py", line 204, in asyncio_run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/home/matt/mamb

Working directory: /home/matt/code/dask_testing/RBFE_tutorial_refactor/production/SOMD/ejm42~ejm54/bound
Using molecules ligand_1, ligand_2, protein:
<BioSimSpace.Molecule: number=3, nAtoms=35, nResidues=1> <BioSimSpace.Molecule: number=44357, nAtoms=39, nResidues=1> <BioSimSpace.Molecule: number=2, nAtoms=4644, nResidues=288>
Mapping..
Aligning..
Merging..
Working directory: /home/matt/code/dask_testing/RBFE_tutorial_refactor/production/SOMD/ejm45~ejm53/bound
Using molecules ligand_1, ligand_2, protein:
<BioSimSpace.Molecule: number=88712, nAtoms=40, nResidues=1> <BioSimSpace.Molecule: number=133066, nAtoms=42, nResidues=1> <BioSimSpace.Molecule: number=88711, nAtoms=4644, nResidues=288>
Mapping..
Aligning..
Merging..


Process Dask Worker process (from Nanny):
Traceback (most recent call last):
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/process.py", line 202, in _run
    target(*args, **kwargs)
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/nanny.py", line 1023, in _run
    asyncio_run(run(), loop_factory=get_loop_factory())
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/compatibility.py", line 204, in asyncio_run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
       

Working directory: /home/matt/code/dask_testing/RBFE_tutorial_refactor/production/SOMD/ejm54~ejm31/bound
Using molecules ligand_1, ligand_2, protein:
<BioSimSpace.Molecule: number=3, nAtoms=39, nResidues=1> <BioSimSpace.Molecule: number=44357, nAtoms=32, nResidues=1> <BioSimSpace.Molecule: number=2, nAtoms=4644, nResidues=288>
Mapping..
Aligning..
Merging..
Working directory: /home/matt/code/dask_testing/RBFE_tutorial_refactor/production/SOMD/ejm52~ejm44/bound
Using molecules ligand_1, ligand_2, protein:
<BioSimSpace.Molecule: number=88713, nAtoms=39, nResidues=1> <BioSimSpace.Molecule: number=133067, nAtoms=38, nResidues=1> <BioSimSpace.Molecule: number=88712, nAtoms=4644, nResidues=288>
Mapping..
Aligning..
Merging..


Process Dask Worker process (from Nanny):
Traceback (most recent call last):
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/process.py", line 202, in _run
    target(*args, **kwargs)
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/nanny.py", line 1023, in _run
    asyncio_run(run(), loop_factory=get_loop_factory())
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/site-packages/distributed/compatibility.py", line 204, in asyncio_run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/home/matt/mambaforge/envs/sireDEV/lib/python3.11/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
       

Working directory: /home/matt/code/dask_testing/RBFE_tutorial_refactor/production/SOMD/ejm31~ejm54/bound
Using molecules ligand_1, ligand_2, protein:
<BioSimSpace.Molecule: number=3, nAtoms=32, nResidues=1> <BioSimSpace.Molecule: number=44358, nAtoms=39, nResidues=1> <BioSimSpace.Molecule: number=2, nAtoms=4644, nResidues=288>
Mapping..
Aligning..
Merging..
Working directory: /home/matt/code/dask_testing/RBFE_tutorial_refactor/production/SOMD/ejm44~ejm52/bound
Using molecules ligand_1, ligand_2, protein:
<BioSimSpace.Molecule: number=88713, nAtoms=38, nResidues=1> <BioSimSpace.Molecule: number=133067, nAtoms=39, nResidues=1> <BioSimSpace.Molecule: number=88712, nAtoms=4644, nResidues=288>
Mapping..
Aligning..
Merging..
